# Generation: Stuffing Documents

In [17]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

In [18]:
embeddings = OllamaEmbeddings(model = "nomic-embed-text")

vectorstore = Chroma(
    persist_directory='./intro-to-ds-lectures',
    embedding_function=embeddings
)

In [19]:
len(vectorstore.get()['documents'])

22

In [20]:
retriever = vectorstore.as_retriever(
    search_type = 'mmr',
    search_kwargs = {
        'k' : 3,
        'lambda_mult' : 0.7
    }
)

In [21]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [22]:
chat = ChatOpenAI(
    model_name='llama3.2:3b',
    openai_api_key='ollama', 
    openai_api_base='http://localhost:11434/v1',
    temperature = 0, 
    max_tokens = 250,
    model_kwargs = {
        'seed':365
    }
)

/opt/anaconda3/envs/ai-ml/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3519: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [23]:
question = "What software do data scientists use?"

In [24]:
chain = {
    'context': retriever,
    'question': RunnablePassthrough()
} | prompt_template

In [25]:
result = chain.invoke(question)
result

StringPromptValue(text="\nAnswer the following question:\nWhat software do data scientists use?\n\nTo answer the question, use only the following context:\n[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!'), Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for 

In [26]:
print(result.text)


Answer the following question:
What software do data scientists use?

To answer the question, use only the following context:
[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!'), Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical 